In [4]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/ETHARTIC/Class-activities/main/Actividades-de-clase/data/datos_de_calidad_de_agua_en_playas.csv"


df = pd.read_csv(csv_url)

# 5. Inspección inicial de los datos
print(f"--- Archivo cargado exitosamente desde: {csv_url.split('/')[-1]} ---")
print(df.info())
print("\nPrimeras 5 filas:")
print(df.head())

## 5.1. Dimensiones y estructura de los datos (Equivalente a str(df))
df.info()
# Nota: df.shape te da exactamente las dimensiones (filas, columnas)
df.shape

# 5.2. Filas únicas: detectando la columna que contiene el ID (Equivalente a length(unique(...)))
# Check for unique values, handle potential missing 'matricula_letra' column from new dataset
if "matricula_letra" in df.columns:
    print(f"Unique values in 'matricula_letra': {df['matricula_letra'].nunique()}")
else:
    print("Column 'matricula_letra' not found in the new dataset. Displaying unique counts for all columns.")

# Si df["columna"].nunique() es igual a len(df), esa columna es un ID único
print("\nValores únicos por columna (Buscar candidatos a ID):")
print(df.nunique())

# 5.3 Datos faltantes

df.isnull().sum()  # Conteo de NA por columna
# (df == "").sum()   # Falsos nulos (cadenas vacías) - This might not be relevant for a new dataset, commenting out for now

## 5.4 Duplicados
df.duplicated().sum() # Cantidad de filas exactamente iguales

## 5.5. Primeras filas
print("\nPrimeras 5 filas:")
print(df.head())

--- Archivo cargado exitosamente desde: datos_de_calidad_de_agua_en_playas.csv ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27166 entries, 0 to 27165
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fecha_muestra         27166 non-null  object 
 1   desc_abreviada_punto  27166 non-null  object 
 2   desc_lugar_muestreo   27166 non-null  object 
 3   playa                 27166 non-null  object 
 4   coliformes            26606 non-null  float64
 5   salinidad             27061 non-null  float64
 6   temperatura           9351 non-null   float64
 7   enterococos           19408 non-null  float64
 8   cianobacterias        27112 non-null  object 
 9   turbidez              19082 non-null  float64
 10  ph                    1532 non-null   float64
 11  tipo                  27166 non-null  object 
dtypes: float64(6), object(6)
memory usage: 2.5+ MB
None

Primeras 5 filas:
      fecha_muestr

In [6]:
total_rows = len(df)

analysis_de_columnas = []

for col in df.columns:
    num_unique = df[col].nunique()
    num_non_null = df[col].count()

    # Determinar si la columna es un posible identificador único
    Columna_Unique = (num_unique == total_rows) and (num_non_null == total_rows)

    analysis_de_columnas.append({
        'Columna': col,
        'Valores_Unicos': num_unique,
        'No_Nulos': num_non_null,
        'Es_ID_Potencial': Columna_Unique
    })


analysis_df = pd.DataFrame(analysis_de_columnas)

display(analysis_df)


,Columna,Valores_Unicos,No_Nulos,Es_ID_Potencial
0,fecha_muestra,26611,27166,False
1,desc_abreviada_punto,23,27166,False
2,desc_lugar_muestreo,22,27166,False
3,playa,21,27166,False
4,coliformes,533,26606,False
5,salinidad,368,27061,False
6,temperatura,233,9351,False
7,enterococos,398,19408,False
8,cianobacterias,3,27112,False
9,turbidez,1077,19082,False


In [12]:
#Verificación de Claves Primarias Simples

# Verificar si alguna columna individual es ya un ID potencial
individual_candidato_PK = analysis_df[analysis_df['Es_ID_Potencial'] == True]

if not individual_candidato_PK.empty:
    print(f"clave primaria de columnas unicas\nColumnas: {individual_candidato_PK['Columna'].tolist()}")
else:
    print("No se encontro una clave primaria de columna unica entre las columnas existentes.")

#idea de buscar el primary key a traves de la combinacion de dos columnas fue de gemini
#elejimos las dos columnas de 'fecha_muestra', y 'desc_abreviada_punto' porque en el contexto de un muestreo, es importante determinar y saber cuando se tomo y donde se tomo la data.
Combinacion_de_columnas_para_chekear = ['fecha_muestra', 'desc_abreviada_punto']
Combinacion_unica = df[Combinacion_de_columnas_para_chekear].drop_duplicates().shape[0] == df.shape[0]

if Combinacion_unica:
    print(f"Las columnas {Combinacion_de_columnas_para_chekear} forman una clave primaria compuesta valida.")
else:
    print(f"Las columnas {Combinacion_de_columnas_para_chekear} NO forman una clave primaria compuesta. Hay filas duplicadas o nulas en esta combinacion.")
    # Para ver los duplicados si existen:
    display(df[df.duplicated(subset=Combinacion_de_columnas_para_chekear, keep=False)].sort_values(by=Combinacion_de_columnas_para_chekear))

No se encontro una clave primaria de columna unica entre las columnas existentes.
Las columnas ['fecha_muestra', 'desc_abreviada_punto'] forman una clave primaria compuesta válida.


Analisis de Salinidad por Playa
(analisis extra por curiosidad)

In [2]:
# calcular la salinidad promedio por playa
salinidad_promedio_playa = df.groupby('playa')['salinidad'].mean().reset_index()

# rankear las playas por salinidad promedio (de menor a mayor salinidad)
salinidad_promedio_playa = salinidad_promedio_playa.sort_values(by='salinidad', ascending=True)

# crear un ranking
salinidad_promedio_playa['ranking_salinidad'] = salinidad_promedio_playa['salinidad'].rank(ascending=True, method='dense')

display(salinidad_promedio_playa.head())


,playa,salinidad,ranking_salinidad
9,La Colorada,4.720660,1.0
15,Punta Espinillo,5.180492,2.0
12,Pajas Blancas,6.044129,3.0
20,Zabala,6.803406,4.0
4,De Los Cilindros,7.379205,5.0
